# 🧭 Árvore Geradora Mínima — Algoritmos (Greedy, Arestas Seguras, Prim e Kruskal)

Este notebook complementa o estudo de AGM com a formulação gulosa, o conceito de **aresta segura** via **cortes**, e a derivação dos algoritmos de **Prim** e **Kruskal**, incluindo visualizações e validações.

## 🛰️ Aplicação
Projeto de redes (comunicações, energia, estradas) conectando n localidades com menor custo total.
- Modelo: grafo não dirigido e ponderado $G=(V,E,w)$.
- Objetivo: encontrar $T\subseteq E$ que conecte todos os vértices minimizando $w(T)$.
- Como $H=(V,T)$ é conectado e acíclico, $H$ é uma **árvore geradora mínima (AGM)**.

## ⚙️ Algoritmo Genérico (Greedy)
Mantém um conjunto $S$ de arestas que está contido em alguma AGM. Em cada iteração, escolhe-se uma **aresta segura** e a adiciona a $S$. Repete até $|S|=|V|-1$.

Intuição: a decisão local (gulosa) é justificável por um teorema com base em **cortes** do grafo.

## ✂️ Definições de Corte e Aresta Segura
- Corte: partição $(V', V\V')$ de $V$.
- Aresta cruza o corte: $(u,v)\in E$ com $u\in V'$ e $v\in V\V'$.
- Corte respeita $S$: nenhuma aresta de $S$ cruza o corte.
- Aresta leve do corte: aresta que cruza o corte com menor peso.

Teorema (aresta segura): Seja $S$ um subconjunto de arestas contido em alguma AGM. Se um corte que respeita $S$ é considerado e $(u,v)$ é uma aresta leve que cruza esse corte, então $(u,v)$ é **segura** para $S$ (isto é, existe uma AGM que contém $S\cup\{(u,v)\}$).

## 🧠 Consequências Práticas
- Kruskal: considera componentes (cortes induzidos pela floresta atual). A aresta leve entre componentes distintas é segura.
- Prim: considera o corte entre a árvore atual e o restante. A aresta leve que conecta a árvore a um novo vértice é segura.

In [ ]:
import heapq
import networkx as nx
import matplotlib.pyplot as plt
from typing import Any, Dict, List, Tuple, Iterable, Set

class UnionFind:
    def __init__(self, elements: Iterable[Any]):
        self.parent = {x: x for x in elements}
        self.rank = {x: 0 for x in elements}
    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, x, y) -> bool:
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        if self.rank[rx] < self.rank[ry]:
            self.parent[rx] = ry
        elif self.rank[rx] > self.rank[ry]:
            self.parent[ry] = rx
        else:
            self.parent[ry] = rx
            self.rank[rx] += 1
        return True

def kruskal_mst(G: nx.Graph):
    if G.is_directed():
        raise ValueError('Kruskal requer grafo não dirigido')
    edges = sorted((data.get('weight',1.0), u, v) for u,v,data in G.edges(data=True))
    uf = UnionFind(G.nodes())
    mst = []
    for w,u,v in edges:
        if uf.union(u,v):
            mst.append((u,v,w))
            if len(mst) == G.number_of_nodes()-1:
                break
    return mst

def prim_mst(G: nx.Graph, start=None):
    if G.is_directed():
        raise ValueError('Prim requer grafo não dirigido')
    if start is None:
        start = next(iter(G.nodes()))
    visited: Set[Any] = set([start])
    heap = []
    for v, data in G[start].items():
        heapq.heappush(heap, (data.get('weight',1.0), start, v))
    mst = []
    while heap and len(visited) < G.number_of_nodes():
        w, u, v = heapq.heappop(heap)
        if v in visited:
            continue
        visited.add(v)
        mst.append((u,v,w))
        for x, data in G[v].items():
            if x not in visited:
                heapq.heappush(heap, (data.get('weight',1.0), v, x))
    return mst

def draw_cut_and_light_edge(G: nx.Graph, cut_set: Set[Any]):
    pos = nx.spring_layout(G, seed=4)
    left = set(cut_set)
    right = set(G.nodes()) - left
    # arestas que cruzam o corte
    crossing = []
    for u,v,data in G.edges(data=True):
        if (u in left and v in right) or (v in left and u in right):
            crossing.append((u,v,data.get('weight',1.0)))
    if not crossing:
        print('Corte não possui arestas cruzando.')
        return
    light = min(crossing, key=lambda x: x[2])
    colors = []
    for u,v in G.edges():
        w = G[u][v].get('weight',1.0)
        if (u==light[0] and v==light[1]) or (u==light[1] and v==light[0]):
            colors.append('crimson')
        elif (u in left and v in right) or (v in left and u in right):
            colors.append('orange')
        else:
            colors.append('gray')
    node_colors = ['lightgreen' if n in left else 'lightblue' for n in G.nodes()]
    fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True, constrained_layout=True)
    nx.draw(G, pos, with_labels=True, node_color=node_colors, node_size=900, font_size=12, edge_color=colors, width=3, ax=ax)
    labels = {(u,v): f'{data.get(
,1.0):.1f}' for u,v,data in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_color='darkgreen', ax=ax)
    ax.set_title('Corte (verde/azul) — arestas cruzando (laranja), aresta leve (vermelha)')
    plt.show()
    return light

## 🧪 Demonstração prática: aresta leve e escolha gulosa
Construímos um grafo ponderado, marcamos um corte, destacamos a **aresta leve** e verificamos que ela coincide com a aresta escolhida pelo passo correspondente de Prim/Kruskal (quando o corte respeita o conjunto atual de arestas).

In [ ]:
G = nx.Graph()
G.add_weighted_edges_from([
    ('A','B',4), ('A','H',8), ('B','H',11), ('B','C',8), ('C','I',2), ('C','F',4),
    ('C','D',7), ('D','E',9), ('D','F',14), ('E','F',10), ('H','I',7), ('H','G',1),
    ('G','I',6), ('G','F',2)
])
# Simular corte de Prim: árvore atual = {'A'}
cut = {'A'}
light = draw_cut_and_light_edge(G, cut)
print('Aresta leve no corte:', light)
mst_prim = prim_mst(G, start='A')
print('Primeiro passo do Prim:', mst_prim[0])
# Eles devem coincidir (mesma aresta, possivelmente invertida)

## ⏱️ Complexidade
- Prim (heap): O(|E| log |V|).
- Kruskal: O(|E| log |E|) ≈ O(|E| log |V|).
- Ambos resultam da escolha gulosa justificável via cortes e arestas seguras.

## 🧩 Exercícios
1. Mostre empiricamente (vários grafos aleatórios) que a aresta escolhida no primeiro passo do Prim é a aresta leve que cruza o corte (Árvore atual, Resto).
2. No Kruskal, ao unir componentes, identifique o corte induzido e verifique que a próxima aresta escolhida é a leve que cruza esse corte.
3. Prove a correção do teorema da aresta segura usando troca (exchange argument).
4. Compare pesos totais obtidos por Prim, Kruskal e NetworkX e discuta discrepâncias (se houver).
5. Modele uma rede real (cidades, custos) e mostre a AGM passo a passo destacando cortes e arestas seguras.